# Week 6: PostGIS: spatial SQL fundamentals

**Track:** Orbital Analyst (Intermediate)  
**Full primer + quiz:** [https://launchdetect.com/academy/week/6/](https://launchdetect.com/academy/week/6/)  
**Track index:** [https://launchdetect.com/academy/orbital-analyst/](https://launchdetect.com/academy/orbital-analyst/)

---

_PostGIS turns PostgreSQL into a full GIS engine. Spatial data types, ST_ functions, GIST indexes — the toolkit for serious geospatial backends._


## Why this week matters

Spreadsheets and notebooks are great until your dataset crosses a million rows or your team grows past three people. Then you need a database. **PostGIS is the database every serious geospatial team uses** because it adds first-class spatial types, hundreds of `ST_*` functions, and GIST indexes that make spatial queries sub-millisecond — to PostgreSQL, the most-loved relational database on the planet.

LaunchDetect's production backend is PostGIS. Every detection record, every spaceport geofence, every NOTAM exclusion zone lives in a PostGIS table. The queries you'll write this week are structurally identical to the queries the production system runs every minute. This isn't toy SQL — this is operational SQL.

Fluency here unlocks the rest of the course: Week 27 (production AWS pipelines) writes to PostGIS, Week 29 (REST API design) reads from it, Capstone 5 sits PostGIS in the middle of the entire detection pipeline.


## Learning objectives

By the end of this lab you will be able to:

- Install PostGIS and load a shapefile
- Write SELECT queries using ST_Within, ST_Distance, ST_Intersects
- Create a spatial index (GIST) and explain its impact
- Build a query that finds all launches within 100 km of a coastline


## Setup — and why these dependencies

**PostGIS itself runs as a Docker container** for this lab — `docker run postgis/postgis:16-3.4`. Don't fight a native install if you've never done one; Docker is one command and it works everywhere.

`sqlalchemy + psycopg2-binary` are the Python adapters. We won't use an ORM (SQLAlchemy ORM hides the spatial SQL you need to learn); we'll use raw `text()` queries through the engine.


In [ ]:
# Install required packages
!pip install -q leafmap[common] skyfield geopandas shapely


## The approach (and why)

Load two tables into PostGIS: a Natural Earth coastlines polygon table and a synthetic detections point table. Write the spatial SQL query that finds every detection within 100 km of any coastline, sorted by distance.

**Why this query specifically?** It exercises every PostGIS pattern you'll need:

1. `ST_DWithin(a::geography, b::geography, 100000)` — geodesic distance lookup in meters. The `::geography` cast is the magic: it tells PostGIS to use accurate ellipsoidal math instead of planar.
2. `ST_Distance(a::geography, b::geography)` — actual distance value, in meters, for the result rows.
3. `ORDER BY distance ASC` + `LIMIT n` — standard pagination pattern.
4. `ST_AsGeoJSON(geom)` — turn the result geometry into a string the frontend can parse directly.

**Why a GIST index on the geometry column?** Without it, the query is O(n²) — every detection compared against every coastline segment. With it, the spatial r-tree prunes to O(log n) candidate pairs in milliseconds. Always create the index.


In [ ]:
# Week 6: PostGIS demo — find launches within 100 km of any coastline.
# (Runs SQL against a hosted PostGIS instance. Local: install with `docker run postgis/postgis`.)
import leafmap.foliumap as leafmap

# Sample query (illustrative — replace conn details with your own PostGIS instance):
SQL = """
SELECT d.id, d.vehicle, d.detected_at,
       ST_AsGeoJSON(d.position) AS geom,
       ST_Distance(d.position::geography, c.geom::geography) / 1000 AS distance_km
FROM detections d, coastlines c
WHERE ST_DWithin(d.position::geography, c.geom::geography, 100000)
ORDER BY distance_km ASC
LIMIT 50;
"""
print("Run this SQL against your PostGIS detections + Natural Earth coastlines tables.")
print(SQL)

# Once you have results, plot them:
m = leafmap.Map(center=[28.49, -80.60], zoom=3)
# m.add_geojson(results_fc)  # uncomment after you have query results
m

# TODO: load Natural Earth coastlines into your PostGIS, run the query above,
# plot the 50 closest-to-coast detections.


## What just happened — and why it works

The query as written is the production shape. Read it left to right:

- `SELECT ... ST_AsGeoJSON(d.position) AS geom` — emit each result row as GeoJSON-ready, so the frontend doesn't need to parse PostGIS-specific formats.
- `FROM detections d, coastlines c` — implicit cross-join. The spatial predicate in the WHERE clause prunes the cross-join down to the spatially-close pairs.
- `WHERE ST_DWithin(d.position::geography, c.geom::geography, 100000)` — the magic line. PostGIS uses the geography cast to compute geodesic distance, the 100000 is meters (100 km), and the GIST index prunes the candidate set to a manageable number before the actual distance is computed.
- `ORDER BY distance_km ASC LIMIT 50` — get the 50 closest, in order. This is the shape of every 'top N nearby' query in any consumer mapping app.

**This pattern generalizes.** Replace `coastlines` with `airports`, `spaceports`, `weather_stations`, and the query structure is identical.


## Common gotchas

- **`::geography` vs `::geometry` is the most-confused thing in PostGIS.** Geography is slower but ellipsoid-accurate. Geometry is faster but planar. Use geography for any global query; use geometry only when you're inside a single projected CRS and need speed.
- **Always create a GIST index on geometry columns.** `CREATE INDEX idx_name ON table USING GIST (column);`. Without it, every query is a full scan.
- **`ST_Distance` units depend on SRID.** For SRID 4326 (WGS84 lat/lon) the planar distance is in degrees, which is meaningless. Cast to geography to get meters.


## Self-check

Verify your solution against these checks before considering the lab complete:

- [ ] Output type matches the expected format (GeoJSON / PNG / table / etc.).
- [ ] No exceptions raised on a clean run.
- [ ] Result is visually plausible — open the map cell, scan the values, sanity-check magnitudes.
- [ ] Quiz on the [week page](https://launchdetect.com/academy/week/6/) — try answering before checking the key.

---

Found a bug or want to contribute an improvement? Open an issue or PR at [github.com/launchdetect/academy-labs](https://github.com/launchdetect/academy-labs).
